# Make `processed/` (INCLUDE-50): download -> split -> frames -> Drive
Builds the shared data folder every member trains on. **CPU runtime is enough** (Runtime -> Change runtime type -> CPU).

Steps: download INCLUDE zips from Zenodo (one at a time, only the 50 classes are kept) -> split -> cache frames -> copy `processed/` to Google Drive.

Work happens on Colab's **local disk** (`/content/isl_data`), because free Google Drive has only 15 GB and the raw videos are large. Only the small `processed/` folder is copied to Drive at the end.
**If the runtime disconnects, the local disk is wiped: start again from cell 0.**

## 0. Setup

In [ ]:
REPO = "https://github.com/Rani553/indian-sign-language-video-recognition.git"
BRANCH = "feature/cnn-lstm"   # has the INCLUDE-50 class list + class-name matching fix; use "develop" once merged
import os
if not os.path.exists("indian-sign-language-video-recognition"):
    !git clone -q $REPO
%cd indian-sign-language-video-recognition
!git fetch -q && git checkout -q $BRANCH && git pull -q
!git log --oneline -3
!bash scripts/setup_colab.sh

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%env ISL_DATA_ROOT=/content/isl_data
%env ISL_RESULTS_ROOT=/content/isl_results
!df -h /content | tail -1
!wc -l configs/include50_classes.txt

## 1. Download
Only the videos of the 50 classes are downloaded (not the whole 57 GB), several zips at a time. Zips are read over the network and never stored.
Watch the log: each finished zip prints `Extracted N videos ... (X/44 zips done)`.

In [ ]:
!python -m src.data.download --list | tail -50

In [ ]:
# Reads each zip's file list from Zenodo and downloads only the 50 classes' videos, 8 zips at a time.
# Safe to re-run: finished zips are skipped and finished videos are not downloaded again.
!python -m src.data.download --remote --workers 8

### Check the class folders
The official list uses joined words (`thankyou`); the folders may be spelled differently. Look at the folder names here.

In [ ]:
!find /content/isl_data/raw -mindepth 2 -maxdepth 2 -type d | sort | head -80
!find /content/isl_data/raw -type f | wc -l
!du -sh /content/isl_data/raw

## 2. Split
Read the log: a warning like `N classes from the list are missing` means some of the 50 names did not match a folder. Fix that before continuing.

In [ ]:
!python -m src.data.split
import json
lm = json.load(open("/content/isl_data/processed/splits/label_map.json"))
print("classes:", len(lm))
assert len(lm) == 50, "Expected 50 classes - check the 'missing' warning above"

## 3. Cache frames (24 frames per video, 224x224)

In [ ]:
!python -m src.features.extract_frames --set frames.workers=2

In [ ]:
import glob, numpy as np, pandas as pd
sp = "/content/isl_data/processed/splits"
total = sum(len(pd.read_csv(f"{sp}/{s}.csv")) for s in ("train", "val", "test"))
files = glob.glob("/content/isl_data/processed/frames/*.npy")
print({s: len(pd.read_csv(f"{sp}/{s}.csv")) for s in ("train", "val", "test")}, "| frames cached:", len(files), "of", total)
a = np.load(files[0]); print(a.shape, a.dtype)
assert len(files) == total and a.shape == (24, 224, 224, 3)

## 4. Copy to Google Drive
`processed/` goes to `MyDrive/isl_data/processed`, which is exactly where the other notebooks look by default (Colab, Drive mounted).

In [ ]:
import shutil, os
dst = "/content/drive/MyDrive/isl_data/processed"
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copytree("/content/isl_data/processed", dst, dirs_exist_ok=True)
!du -sh $dst

## 5. Share with the team
* **Colab users:** in Google Drive, right-click the folder `isl_data` -> Share -> add teammates (Viewer). Each teammate: right-click -> *Add shortcut to Drive* -> My Drive.
* **Kaggle users:** make one zip and upload it as a Kaggle Dataset (e.g. `isl-include50-processed`), then use `--set paths.processed_dir=/kaggle/input/isl-include50-processed`.

In [ ]:
# Optional: one zip file for a Kaggle Dataset (stored frames are already compressed .npy, so this is mostly packaging)
!cd /content/isl_data && zip -qr /content/drive/MyDrive/isl_data/processed.zip processed
!ls -lh /content/drive/MyDrive/isl_data/processed.zip